# LAVIS Image Analysis - Colab Notebook

In [ ]:
# Step 2: Imports
import sys
sys.path.insert(0, '.')

import torch
from PIL import Image
from IPython.display import display
import requests
from io import BytesIO
from lavis.models import load_model_and_preprocess

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# Step 3: Load image
url = "https://raw.githubusercontent.com/salesforce/LAVIS/main/docs/_static/merlion.png"
raw_image = Image.open(BytesIO(requests.get(url).content)).convert("RGB")
display(raw_image)

In [ ]:
# Step 4: Load captioning model
model, vis_processors, _ = load_model_and_preprocess(
    name="blip_caption", model_type="base_coco", is_eval=True, device=device
)

In [ ]:
# Step 5: Generate caption
image = vis_processors["eval"](raw_image).unsqueeze(0).to(device)
caption = model.generate({"image": image})[0]
print("Caption:", caption)

In [ ]:
# Step 6: Load VQA model
vqa_model, vqa_vis, vqa_txt = load_model_and_preprocess(
    name="blip_vqa", model_type="vqav2", is_eval=True, device=device
)

In [ ]:
# Step 7: Ask a question
question = "What is this place?"
img = vqa_vis["eval"](raw_image).unsqueeze(0).to(device)
q = vqa_txt["eval"](question)
answer = vqa_model.predict_answers(
    samples={"image": img, "text_input": q}, inference_method="generate"
)[0]
print("Q:", question)
print("A:", answer)

In [ ]:
# Step 8: Try with your own image (upload)
from google.colab import files
uploaded = files.upload()
fname = list(uploaded.keys())[0]
my_image = Image.open(BytesIO(uploaded[fname])).convert("RGB")
display(my_image)

# Caption your image
img_tensor = vis_processors["eval"](my_image).unsqueeze(0).to(device)
my_caption = model.generate({"image": img_tensor})[0]
print("Your image caption:", my_caption)

## 2. Clone LAVIS Repository and Verify Installation

Cloning the repository is optional if you already installed `salesforce-lavis` from PyPI, but it is useful to browse examples and docs.

In [ ]:
# Colab setup: clone the LAVIS repo and install ALL its dependencies.
import os

# Clone repo into /content/LAVIS (or current working directory in Colab)
if not os.path.exists("LAVIS"):
    !git clone https://github.com/salesforce/LAVIS.git
else:
    print("LAVIS repo already present.")

# Change working directory into the repo
%cd LAVIS
print("Now working inside LAVIS repo at:", os.getcwd())

# Install all dependencies from requirements.txt first
!pip install -q -r requirements.txt

# Then install LAVIS itself in editable mode
!pip install -q -e .

print("LAVIS and all dependencies are installed.")

## 1. Set Up Environment and Install LAVIS

Run this cell to install LAVIS and its dependencies. On Colab this may take several minutes and might restart the kernel.

In [ ]:
# Quick sanity check: show top-level contents of the LAVIS repo
import os

print("Repo root:", os.getcwd())
print("Top-level files and folders:", os.listdir("."))

## 3. Import LAVIS and Supporting Libraries

Import PyTorch, PIL for image handling, and LAVIS utilities. We also set up the computation device (GPU if available).

In [ ]:
# Ensure the current LAVIS repo root is on sys.path so we can import it.
import os
import sys

repo_root = os.path.abspath(".")
if repo_root not in sys.path:
    sys.path.append(repo_root)
    print("Added to sys.path:", repo_root)

import torch
from PIL import Image
from IPython.display import display
import requests
from io import BytesIO

from lavis.models import load_model_and_preprocess

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 4. Download or Upload a Test Image

You can either:
- **Option A (default)**: Download an example image from a URL.
- **Option B**: Upload your own image file in Colab.

Below we show both approaches.

In [ ]:
# Option A: Download an image from the web
image_url = "https://raw.githubusercontent.com/salesforce/LAVIS/main/docs/_static/merlion.png"

response = requests.get(image_url)
raw_image = Image.open(BytesIO(response.content)).convert("RGB")

print("Downloaded image size:", raw_image.size)
display(raw_image)

In [ ]:
# Option B: Upload your own image (uncomment when running in Colab)
# from google.colab import files
# uploaded = files.upload()
# from PIL import Image
# import io
#
# # Take the first uploaded file
# filename = next(iter(uploaded))
# raw_image = Image.open(io.BytesIO(uploaded[filename])).convert("RGB")
# print("Uploaded image:", filename, "size:", raw_image.size)
# display(raw_image)

## 5. Load a Pretrained BLIP Captioning Model

We now load a BLIP captioning model and its preprocessors using `load_model_and_preprocess`.

In [ ]:
# Load BLIP captioning model (base size, trained on COCO)

caption_model, vis_processors, txt_processors = load_model_and_preprocess(
    name="blip_caption", model_type="base_coco", is_eval=True, device=device
)

print("Loaded captioning model:", type(caption_model))

## 6. Preprocess the Image for the Model

Use the vision preprocessor returned by LAVIS to convert the raw `PIL.Image` into a tensor batch suitable for the model.

In [ ]:
# Preprocess image for evaluation
image_tensor = vis_processors["eval"](raw_image).unsqueeze(0).to(device)
print("Preprocessed image tensor shape:", image_tensor.shape)

## 7. Generate a Caption for the Image

Use the captioning model to generate a textual description of the input image.

In [ ]:
# Generate a caption for the image
with torch.no_grad():
    caption = caption_model.generate({"image": image_tensor})[0]

print("Generated caption:", caption)

## 8. Visual Question Answering (VQA) on the Same Image

We now load a BLIP VQA model, ask a question about the same image, and predict an answer.

In [ ]:
# Load BLIP VQA model
vqa_model, vqa_vis_processors, vqa_txt_processors = load_model_and_preprocess(
    name="blip_vqa", model_type="vqav2", is_eval=True, device=device
)

# Reuse the same image, but preprocess with the VQA vision processor
vqa_image = vqa_vis_processors["eval"](raw_image).unsqueeze(0).to(device)

question = "What is this place?"
question_proc = vqa_txt_processors["eval"](question)

with torch.no_grad():
    answer = vqa_model.predict_answers(
        samples={"image": vqa_image, "text_input": question_proc},
        inference_method="generate",
    )[0]

print("Question:", question)
print("Predicted answer:", answer)

## 9. Wrap Inference in Reusable Helper Functions

To make this notebook easy to reuse, we define functions for captioning and VQA on arbitrary images.

In [ ]:
def generate_caption(pil_image: Image.Image) -> str:
    """Generate a caption for a PIL image using the BLIP captioning model."""
    img_tensor = vis_processors["eval"](pil_image).unsqueeze(0).to(device)
    with torch.no_grad():
        caption_out = caption_model.generate({"image": img_tensor})[0]
    return caption_out


def answer_question(pil_image: Image.Image, question: str) -> str:
    """Answer a question about a PIL image using the BLIP VQA model."""
    img_tensor = vqa_vis_processors["eval"](pil_image).unsqueeze(0).to(device)
    question_proc = vqa_txt_processors["eval"](question)
    with torch.no_grad():
        answer_out = vqa_model.predict_answers(
            {"image": img_tensor, "text_input": [question_proc]}, k=1
        )[0]
    return answer_out


# Demo: use helpers on the current image
print("Helper caption:", generate_caption(raw_image))
print("Helper VQA answer:", answer_question(raw_image, "What animal statue is this?"))

In [ ]:
# Step 1: Clone LAVIS and install all dependencies
!git clone https://github.com/salesforce/LAVIS.git
%cd LAVIS
!pip install -q torch torchvision torchaudio
!pip install -q transformers==4.33.2 timm==0.4.12 fairscale==0.4.4
!pip install -q omegaconf iopath decord webdataset
!pip install -q opencv-python-headless Pillow requests
!pip install -q -e .

In [ ]:
# Step 2: Import libraries
import sys
sys.path.append('/content/LAVIS')

import torch
from PIL import Image
from IPython.display import display
import requests
from io import BytesIO
from lavis.models import load_model_and_preprocess

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
# Step 3: Load test image
image_url = "https://raw.githubusercontent.com/salesforce/LAVIS/main/docs/_static/merlion.png"
raw_image = Image.open(BytesIO(requests.get(image_url).content)).convert("RGB")
display(raw_image)

In [ ]:
# Step 4: Load BLIP captioning model
model, vis_processors, _ = load_model_and_preprocess(
    name="blip_caption", model_type="base_coco", is_eval=True, device=device
)
print("Captioning model loaded.")

In [ ]:
# Step 5: Generate caption
image = vis_processors["eval"](raw_image).unsqueeze(0).to(device)
caption = model.generate({"image": image})[0]
print("Caption:", caption)

In [ ]:
# Step 6: Load BLIP VQA model
vqa_model, vqa_vis_processors, vqa_txt_processors = load_model_and_preprocess(
    name="blip_vqa", model_type="vqav2", is_eval=True, device=device
)
print("VQA model loaded.")

In [ ]:
# Step 7: Ask a question about the image
vqa_image = vqa_vis_processors["eval"](raw_image).unsqueeze(0).to(device)
question = "What is this place?"
question_proc = vqa_txt_processors["eval"](question)

answer = vqa_model.predict_answers(
    samples={"image": vqa_image, "text_input": question_proc},
    inference_method="generate"
)[0]

print("Question:", question)
print("Answer:", answer)

In [ ]:
# Step 8: Try with your own image (upload)
from google.colab import files
uploaded = files.upload()

for filename in uploaded.keys():
    my_image = Image.open(BytesIO(uploaded[filename])).convert("RGB")
    display(my_image)
    
    # Caption
    img_tensor = vis_processors["eval"](my_image).unsqueeze(0).to(device)
    my_caption = model.generate({"image": img_tensor})[0]
    print("Caption:", my_caption)
    
    # VQA
    vqa_tensor = vqa_vis_processors["eval"](my_image).unsqueeze(0).to(device)
    my_question = "What do you see in this image?"
    my_question_proc = vqa_txt_processors["eval"](my_question)
    my_answer = vqa_model.predict_answers(
        samples={"image": vqa_tensor, "text_input": my_question_proc},
        inference_method="generate"
    )[0]
    print("Question:", my_question)
    print("Answer:", my_answer)